In [9]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path

In [10]:
CC_3080 = json.loads(open('./3080_compile_commands.json').read())
CC_A100 = json.loads(open('./a100_compile_commands.json').read())
CC_A10 = json.loads(open('./a10_compile_commands.json').read())

In [11]:
!ls ./3080/*.csv ./3080/*.json
!ls ./A10/*.csv ./A10/*.json
!ls ./A100/*.csv ./A100/*.json

!head -n 2 ./3080/NVIDIA_GeForce_RTX_3080_gpuData.csv

./3080/NVIDIA_GeForce_RTX_3080_gpuData.csv
./3080/NVIDIA_GeForce_RTX_3080_profiling-log-20260126-110925.json
./A10/NVIDIA_A10_gpuData.csv
./A10/NVIDIA_A10_profiling-log-20260126-224615.json
./A100/NVIDIA_A100-SXM4-40GB_gpuData.csv
./A100/NVIDIA_A100-SXM4-40GB_profiling-log-20260126-175538.json
"source","exePath","sample","kernel_executed","eteProfilerXtime","CC","Kernel Name","traffic","bytesRead","bytesWrite","bytesTotal","dpAI","spAI","hpAI","dpPerf","spPerf","hpPerf","xtime","Block Size","Grid Size","device","SP_FLOP","DP_FLOP","HP_FLOP","INTOP","intPerf","intAI","targetName","exeArgs","runtime","kernelMangled","kernelName","kernelDemangled","kernelProfiler"
"accuracy-cuda","/home/gbolet/gpuFLOPBench-updated/build/bin/cuda/accuracy",1,"normal",2.34109780199924,8.6,"_Z15accuracy_kerneliiiPKfPKiPi",645764152717.7,327806976.0,1811712.0,329618688.0,0.0,0.0,0.0,0.0,0.0,0.0,510432.0,"(256, 1, 1)","(2048, 1, 1)","NVIDIA GeForce RTX 3080",0.0,0.0,0.0,562998344.0,1102984029214.4692,1.7080292

In [12]:
og_df3080 = pd.read_csv('./3080/NVIDIA_GeForce_RTX_3080_gpuData.csv')
og_dfA10 = pd.read_csv('./A10/NVIDIA_A10_gpuData.csv')
og_dfA100 = pd.read_csv('./A100/NVIDIA_A100-SXM4-40GB_gpuData.csv')

og_data = pd.concat([og_df3080, og_dfA10, og_dfA100], ignore_index=True)

In [13]:
# there seems to be an issue where the above CSVs are missing some data that actually exists in the ncu-rep files
# in order to have proper data, we will re-parse all the ncu-rep files to build a complete dataframe



reports3080 = [ path.abspath(report) for report in glob.glob('./3080/*.ncu-rep')]
reportsA10 = [ path.abspath(report) for report in glob.glob('./A10/*.ncu-rep')]
reportsA100 = [ path.abspath(report) for report in glob.glob('./A100/*.ncu-rep')]

In [14]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + log_metrics

# markers we should ignore / drop kernels containing these from the dataset
library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [15]:
def _process_single_report(report):
    ncuResult = _parse_ncu_report(report)

    rawDF = roofline_results_to_df(ncuResult)

    # drop any kernel names that contain

    # drop the first row which is units
    rawDF = rawDF.iloc[1:].copy(deep=True)

    if rawDF['dram__bytes_read.sum'].isna().any() or rawDF['dram__bytes_write.sum'].isna().any():
        print(f"  WARNING: NCU report contains NaN values for some metrics for {report}")
        # we need to drop these rows to avoid errors

    rawDF = rawDF[(rawDF['dram__bytes_read.sum'].notna()) & (rawDF['dram__bytes_write.sum'].notna())]

    roofDF = calc_roofline_data(rawDF)

    if roofDF.empty:
        print(f'  No roofline data parsed from {report}')
        return None

    # we need to add the source and modelType columns manually
    # the report file name is in the format of <device>_source-modelType-ssampleNumber-report.ncu-rep
    device_name = roofDF['device'].unique()[0].replace(' ', '_')
    filename = path.basename(report)
    match = re.match(rf'{device_name}_(.+)-(.+)-s(\d+)-report\.ncu-rep', filename)

    source = match.group(1) if match else 'unknown'
    model_type = match.group(2) if match else 'unknown'
    sample = match.group(3) if match else 'unknown'

    assert source != 'unknown', f'Could not parse source from filename {filename}'  
    assert sample != 'unknown', f'Could not parse sample from filename {filename}'
    assert model_type != 'unknown', f'Could not parse model_type from filename {filename}'
    assert model_type in ['cuda', 'omp'], f'Parsed model_type {model_type} is not valid from filename {filename}'

    source_name = source+'-'+model_type
    roofDF['source'] = source_name
    roofDF['model_type'] = model_type
    roofDF['sample'] = sample

    # for some of the CUDA kernels, if we
    if model_type == 'cuda':
        roofDF['Demangled Name'] = roofDF['Kernel Name'].apply(lambda x: demangle_kernel_name(x))
    elif model_type == 'omp':
        roofDF['Demangled Name'] = roofDF['Kernel Name'].apply(lambda x: demangle_omp_offload_name(x))

    # drop any rows that contain library markers
    beforeRows = roofDF.shape[0]
    roofDF = roofDF[~roofDF['Demangled Name'].str.contains('|'.join(library_markers))]
    afterRows = roofDF.shape[0]
    dropped = beforeRows - afterRows
    if dropped > 0:
        print(f'\t  Dropped {dropped} library/kernel rows from {report}')

        if roofDF.empty:
            print(f'  No roofline data remaining after dropping library kernels from {report}')
            return None

    # we need to extract the exeArgs from the og dataframes
    # NOTE: this will break if we execute a code more than once with diff exe args
    exeArgs = extract_exe_args_from_ncu_report(report)
    roofDF['exeArgs'] = ' '.join(exeArgs) if exeArgs else None

    # limit the output columns to the cols we care about
    return roofDF[all_cols]

# had to multithread this operation because single-thread is 30 mins for all reports
def load_ncu_reports(reportsList, max_workers=None, chunksize=1):
    fn = partial(_process_single_report)
    dfs = process_map(fn, reportsList,
                      max_workers=max_workers or os.cpu_count(),
                      chunksize=chunksize,
                      desc=f"Processing {len(reportsList)} reports")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()



In [16]:
testReport = [os.path.abspath('./A100/NVIDIA_A100-SXM4-40GB_aligned-types-omp-s1-report.ncu-rep')]

testdf = load_ncu_reports(testReport)
#print(testdf)

testReport = [os.path.abspath('./3080/NVIDIA_GeForce_RTX_3080_aligned-types-omp-s1-report.ncu-rep')]
testdf = load_ncu_reports(testReport)
#print(testdf)

print()

Processing 1 reports:   0%|          | 0/1 [00:00<?, ?it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/A100/NVIDIA_A100-SXM4-40GB_aligned-types-omp-s1-report.ncu-rep


Processing 1 reports:   0%|          | 0/1 [00:00<?, ?it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/3080/NVIDIA_GeForce_RTX_3080_aligned-types-omp-s1-report.ncu-rep


Processing 1 reports: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]

In [17]:
df3080 = load_ncu_reports(reports3080, 8)
print()
dfA10 = load_ncu_reports(reportsA10, 8)
print()
dfA100 = load_ncu_reports(reportsA100, 8)

Processing 1907 reports:   0%|          | 1/1907 [00:00<15:06,  2.10it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/3080/NVIDIA_GeForce_RTX_3080_ldpc-cuda-s3-report.ncu-rep


Processing 1907 reports:   1%|          | 19/1907 [00:01<02:11, 14.37it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/3080/NVIDIA_GeForce_RTX_3080_jacobi-cuda-s3-report.ncu-rep


Processing 1907 reports:   1%|▏         | 27/1907 [00:02<02:06, 14.85it/s]

...

Processing 1852 reports:  98%|█████████▊| 1818/1852 [02:07<00:02, 13.88it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/A100/NVIDIA_A100-SXM4-40GB_keccaktreehash-cuda-s3-report.ncu-rep


Processing 1852 reports: 100%|█████████▉| 1847/1852 [02:09<00:00, 13.58it/s]

  No arguments found in ncu-rep file /home/gbolet/gpuFLOPBench-updated/cuda-profiling/collected-data/A100/NVIDIA_A100-SXM4-40GB_clenergy-omp-s2-report.ncu-rep


Processing 1852 reports: 100%|██████████| 1852/1852 [02:09<00:00, 14.25it/s]


In [18]:
print(df3080.shape)
print(dfA10.shape)
print(dfA100.shape)

(5417, 24)
(5205, 24)
(4885, 24)


In [18]:
print(df3080.columns)

print(dfA100.columns)

print(set(df3080.columns) ^ set(dfA100.columns))

assert df3080.columns.tolist() == dfA10.columns.tolist() == dfA100.columns.tolist()

Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'device', 'SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'intPerf', 'intAI',
       'source', 'model_type', 'sample'],
      dtype='object', length=978)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'device', 'SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'intPerf', 'intAI',
       'source', 'model_type', 'sample'],
      dtype='object', length=982)
{'sm__pipe_shared_cycles_active.avg.pct_of_peak_sustained_elapsed', 'sm__inst_executed_pipe_fp16.max.pct_of_peak_sustained_elapsed', 'sm__pipe_shared_cycles_active.sum.pct_of_peak_sustained_elapsed', 'sm__pipe_shared_cycles_active.max.pct_of_peak_sustained_elapsed', 'sm__pipe_fmaheavy_cycles_active.max.pct_of_peak_sustained_elapsed', 'sm__inst_executed_pipe_fp16.sum.pct_of_peak_sustained_elaps

AssertionError: 

In [ ]:
# because we manually loaded all the ncu-rep files, we need to add the "source" and "modelType" columns
# manually




In [ ]:
# merge the dataframes into one df

df = pd.concat([df3080, dfA10, dfA100], ignore_index=True)

In [ ]:
# let's drop any samples that are not "normal" -- they either timed out and weren't collected, or never ran
print(df.shape)
#df = df[df['kernel_executed'] == "normal"]
#print(df.shape)

In [ ]:
print(df['device'].value_counts())

In [ ]:
def is_cuda_or_omp_code(row):
    if ('-cuda' in row['source']) and ('/cuda/' in row['exePath']):
        return 'cuda'
    elif ('-omp' in row['source']) and ('/omp/' in row['exePath']):
        return 'omp'
    else:
        raise Exception('target code is neither OpenMP nor CUDA!')

# infer model type from source/exePath
df['model_type'] = df.apply(lambda row: is_cuda_or_omp_code(row), axis=1)

# we need to fix the omp Kernel Name columns
def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x
# the rest of the chars are the function name and line number -- which is the same across compilations
df['Kernel Name'] = df['Kernel Name'].apply(fix_omp_kernel_name)

In [ ]:
testdf = df[df['Kernel Name'] == '_Z7runTestI13uint3_alignedEiPhS1_ii_l196'][['device', 'kernel_executed', 'Kernel Name', 'model_type', 'source', 'exePath']]

display(testdf)

# noting that for some weird reason, this particular kernel gets sampled multiple times per device
# not sure exactly why, but it's of no bother. The real problem is that we seem to be missing the A100 data for this kernel.

In [ ]:
print(df[df['model_type']=='omp'][['model_type', 'Kernel Name', 'device']].sort_values(['model_type', 'Kernel Name', 'device']).head(25))

print(df['model_type'].value_counts())



In [ ]:

df = df[all_cols]

In [ ]:
# drop any rows with NaN in any of the metrics
df = df.dropna(subset=metrics+log_metrics)

# make sure that for each key_cols and device_col grouping, when we mean the metrics, that we have exactly 3 samples (one per device)
# if we don't have 3 devices per group, drop the group and print a warning
grouped = df.groupby(key_cols)
for name, group in grouped:
    new_grp = group.groupby('device')[metrics].mean()
    gpu_data_list = new_grp.index.tolist()
    #print(f'Group: {name}, devices: {new_grp.index.tolist()}')
    if len(gpu_data_list) != 3:
        print(f'\tWarning: dropping group {name} with {new_grp.shape[0]} samples (expected 3)')
        df = df.drop(group.index)

In [ ]:
# Dataset quick look for metrics (non-null counts and ranges)
metric_summary = df[metrics].apply(pd.to_numeric, errors='coerce').describe().T
display(metric_summary[['count','min','max','mean','50%']])

# Plot settings
sns.set_theme(style='whitegrid')

def _plot_metric_side(ax, data, metric, title):
    # Ensure numeric + positive for log scale
    data = data.copy()
    data[metric] = pd.to_numeric(data[metric], errors='coerce')
    data = data[np.isfinite(data[metric]) & (data[metric] >= 0)]
    device_order = sorted(data['device'].unique())
    x_pos = {d: i for i, d in enumerate(device_order)}
    # scatter points
    sns.stripplot(
        data=data,
        x='device',
        y=metric,
        order=device_order,
        jitter=0.1,
        size=3,
        alpha=0.7,
        ax=ax
    )
    # connect matching kernels across devices 
    lines_df = (
        data.groupby(key_cols, dropna=False)[metric]
        .mean()
        .reset_index()
    )

    print('Lines DF for metric:', metric)
    print(lines_df.head())

    for _, grp in lines_df.groupby(key_cols, dropna=False):
        print('Group:', grp)
        # assert that the group has exactly 3 entries (one per device)
        if len(grp) != 3:
            raise Exception(f'Expected 3 entries per group, got {len(grp)}')
        grp = grp.sort_values(['device'])
        xs = [x_pos[d] for d in grp['device']]
        ys = grp[metric].tolist()
        if len(xs) > 1:
            ax.plot(xs, ys, color='gray', alpha=0.6, linewidth=0.8, zorder=1)
    ax.set_title(title)
    ax.set_yscale('symlog')
    ax.set_xlabel('device')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

def plot_metric(metric):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    _plot_metric_side(
        axes[0],
        df[df['model_type'] == 'omp'],
        metric,
        f'OpenMP: {metric}'
    )
    _plot_metric_side(
        axes[1],
        df[df['model_type'] == 'cuda'],
        metric,
        f'CUDA: {metric}'
    )
    fig.tight_layout()
    return fig

# Plot all metrics of interest (skip metrics with no positive values)
for metric in metrics:
    if (pd.to_numeric(df[metric], errors='coerce') > 0).any():
        plot_metric(metric)
        plt.show()
    break